# Brazilian E-Commerce Analysis — Olist Dataset
## Phase 1: Data Cleaning & Merging
**Author:** Yasser Ramzy  
**Dataset:** [Olist Brazilian E-Commerce — Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Tools:** Python · Pandas · NumPy  

### Project Overview
This notebook covers the full data cleaning and merging pipeline for the Olist dataset.
The dataset contains 9 relational tables covering 100k+ real e-commerce orders in Brazil (2016–2018).
The goal is to produce a single clean master table ready for analysis, SQL querying, and Power BI dashboarding.


In [2]:
# ── Libraries ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


## Step 1: Load All Datasets
We load all 9 CSV files from the data folder and preview their shape and structure.

In [3]:
# ── File path ───────────────────────────────────────────────
path = r"C:\Users\yasse\OneDrive\Desktop\data analysis projects\Brazilian-E-Commerce-SQL-POWERBI\Brazilian-Olist-SQL-POWERBI-finished-project\data"

# ── Load all 9 tables ───────────────────────────────────────
orders        = pd.read_csv(os.path.join(path, 'olist_orders_dataset.csv'))
order_items   = pd.read_csv(os.path.join(path, 'olist_order_items_dataset.csv'))
order_pays    = pd.read_csv(os.path.join(path, 'olist_order_payments_dataset.csv'))
order_reviews = pd.read_csv(os.path.join(path, 'olist_order_reviews_dataset.csv'))
customers     = pd.read_csv(os.path.join(path, 'olist_customers_dataset.csv'))
sellers       = pd.read_csv(os.path.join(path, 'olist_sellers_dataset.csv'))
products      = pd.read_csv(os.path.join(path, 'olist_products_dataset.csv'))
geo           = pd.read_csv(os.path.join(path, 'olist_geolocation_dataset.csv'))
category_tr   = pd.read_csv(os.path.join(path, 'product_category_name_translation.csv'))

print("✅ All datasets loaded")
print(f"\n{'Table':<20} {'Rows':>8} {'Columns':>8}")
print("-" * 38)
for name, df in [('orders', orders), ('order_items', order_items),
                 ('order_pays', order_pays), ('order_reviews', order_reviews),
                 ('customers', customers), ('sellers', sellers),
                 ('products', products), ('geo', geo),
                 ('category_tr', category_tr)]:
    print(f"{name:<20} {df.shape[0]:>8,} {df.shape[1]:>8}")

✅ All datasets loaded

Table                    Rows  Columns
--------------------------------------
orders                 99,441        8
order_items           112,650        7
order_pays            103,886        5
order_reviews          99,224        7
customers              99,441        5
sellers                 3,095        4
products               32,951        9
geo                  1,000,163        5
category_tr                71        2


## Step 2: Explore Each Table
We inspect data types, missing values, and key columns for each table before merging.

In [4]:
# ── Reusable function to profile any dataframe ──────────────
def profile(df, name):
    print(f"\n{'='*50}")
    print(f"  TABLE: {name}")
    print(f"{'='*50}")
    print(f"  Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\n  {'Column':<35} {'Dtype':<15} {'Nulls':>8} {'Null%':>8}")
    print(f"  {'-'*70}")
    for col in df.columns:
        nulls = df[col].isnull().sum()
        pct   = nulls / len(df) * 100
        print(f"  {col:<35} {str(df[col].dtype):<15} {nulls:>8,} {pct:>7.1f}%")
    print()

# ── Profile all tables ──────────────────────────────────────
for name, df in [('orders', orders), ('order_items', order_items),
                 ('order_pays', order_pays), ('order_reviews', order_reviews),
                 ('customers', customers), ('sellers', sellers),
                 ('products', products), ('category_tr', category_tr)]:
    profile(df, name)


  TABLE: orders
  Shape : 99,441 rows × 8 columns

  Column                              Dtype              Nulls    Null%
  ----------------------------------------------------------------------
  order_id                            object                 0     0.0%
  customer_id                         object                 0     0.0%
  order_status                        object                 0     0.0%
  order_purchase_timestamp            object                 0     0.0%
  order_approved_at                   object               160     0.2%
  order_delivered_carrier_date        object             1,783     1.8%
  order_delivered_customer_date       object             2,965     3.0%
  order_estimated_delivery_date       object                 0     0.0%


  TABLE: order_items
  Shape : 112,650 rows × 7 columns

  Column                              Dtype              Nulls    Null%
  ----------------------------------------------------------------------
  order_id             

## Step 3: Data Cleaning
We fix data types, handle missing values, and clean each table before merging.

In [5]:
# ── Clean: orders ───────────────────────────────────────────
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("✅ orders — date columns converted")
print(orders[date_cols].dtypes)

✅ orders — date columns converted
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [6]:
# ── Clean: products ─────────────────────────────────────────

# Fill missing category with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Fill missing dimensions/weight with median
dim_cols = ['product_weight_g', 'product_length_cm',
            'product_height_cm', 'product_width_cm']
for col in dim_cols:
    products[col] = products[col].fillna(products[col].median())

# Add English category by merging translation table
products = products.merge(category_tr, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

# Drop columns we won't need in analysis
products.drop(columns=['product_name_lenght',
                        'product_description_lenght',
                        'product_photos_qty'], inplace=True)

print("✅ products — cleaned")
print(f"   Shape: {products.shape}")
print(f"   Nulls remaining:\n{products.isnull().sum()}")

✅ products — cleaned
   Shape: (32951, 7)
   Nulls remaining:
product_id                       0
product_category_name            0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
dtype: int64


In [7]:
# ── Clean: order_reviews ────────────────────────────────────

# Convert dates
order_reviews['review_creation_date']    = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])

# Fill comment nulls with empty string (absence of comment is valid data)
order_reviews['review_comment_title']   = order_reviews['review_comment_title'].fillna('')
order_reviews['review_comment_message'] = order_reviews['review_comment_message'].fillna('')

# Keep only one review per order (some orders have duplicates — keep latest)
order_reviews = order_reviews.sort_values('review_answer_timestamp')\
                              .drop_duplicates(subset='order_id', keep='last')

print("✅ order_reviews — cleaned")
print(f"   Shape after dedup: {order_reviews.shape}")

✅ order_reviews — cleaned
   Shape after dedup: (98673, 7)


In [8]:
# ── Clean: order_payments ───────────────────────────────────

# Aggregate payments per order
# (some orders have multiple payment rows e.g. voucher + credit card)
order_pays_agg = order_pays.groupby('order_id').agg(
    total_payment_value  = ('payment_value',        'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_type         = ('payment_type',         lambda x: x.mode()[0])
).reset_index()

print("✅ order_payments — aggregated")
print(f"   Shape: {order_pays_agg.shape}")
print(f"   Payment types:\n{order_pays['payment_type'].value_counts()}")

✅ order_payments — aggregated
   Shape: (99440, 4)
   Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [9]:
# ── Clean: order_items ──────────────────────────────────────

order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

# Aggregate to order level
# (each row is one item — we want totals per order)
order_items_agg = order_items.groupby('order_id').agg(
    order_total_price    = ('price',          'sum'),
    order_freight_value  = ('freight_value',  'sum'),
    order_items_count    = ('order_item_id',  'max'),
    product_id           = ('product_id',     'first'),
    seller_id            = ('seller_id',      'first')
).reset_index()

print("✅ order_items — aggregated")
print(f"   Shape: {order_items_agg.shape}")

✅ order_items — aggregated
   Shape: (98666, 6)


## Step 4: Merging All Tables
We join all cleaned tables into a single master dataframe using order_id as the primary key.
We use LEFT joins throughout to keep all orders, even those missing payment or review data.

In [10]:
# ── Master merge ────────────────────────────────────────────

# Start with orders as the base (99,441 rows)
master = orders.copy()

# 1. Add customer info
master = master.merge(customers, on='customer_id', how='left')

# 2. Add order items (price, freight, count, product, seller)
master = master.merge(order_items_agg, on='order_id', how='left')

# 3. Add product info (category)
master = master.merge(
    products[['product_id', 'product_category_name_english',
              'product_weight_g', 'product_length_cm',
              'product_height_cm', 'product_width_cm']],
    on='product_id', how='left'
)

# 4. Add seller info
master = master.merge(
    sellers[['seller_id', 'seller_city', 'seller_state']],
    on='seller_id', how='left'
)

# 5. Add payment info
master = master.merge(order_pays_agg, on='order_id', how='left')

# 6. Add review info
master = master.merge(
    order_reviews[['order_id', 'review_score',
                   'review_comment_message', 'review_creation_date']],
    on='order_id', how='left'
)

print("✅ Master table created")
print(f"   Shape: {master.shape}")
print(f"\n   Columns:\n{list(master.columns)}")

✅ Master table created
   Shape: (99441, 30)

   Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_total_price', 'order_freight_value', 'order_items_count', 'product_id', 'seller_id', 'product_category_name_english', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_city', 'seller_state', 'total_payment_value', 'payment_installments', 'payment_type', 'review_score', 'review_comment_message', 'review_creation_date']


## Step 5: Feature Engineering
We create new calculated columns that will power our analysis and dashboard.

In [11]:
# ── Feature Engineering ─────────────────────────────────────

# 1. Delivery time in days (purchase → delivered to customer)
master['delivery_days'] = (
    master['order_delivered_customer_date'] -
    master['order_purchase_timestamp']
).dt.days

# 2. Estimated vs actual delivery difference
master['delivery_delay_days'] = (
    master['order_delivered_customer_date'] -
    master['order_estimated_delivery_date']
).dt.days

# 3. Late delivery flag (1 = late, 0 = on time or early)
master['is_late'] = (master['delivery_delay_days'] > 0).astype(int)

# 4. Total order value (price + freight)
master['order_total_value'] = (
    master['order_total_price'] +
    master['order_freight_value']
)

# 5. Order purchase date parts (for time series analysis)
master['purchase_year']  = master['order_purchase_timestamp'].dt.year
master['purchase_month'] = master['order_purchase_timestamp'].dt.month
master['purchase_month_name'] = master['order_purchase_timestamp'].dt.strftime('%b')
master['purchase_day_of_week'] = master['order_purchase_timestamp'].dt.day_name()

# 6. Freight ratio (freight as % of total order value)
master['freight_ratio'] = (
    master['order_freight_value'] /
    master['order_total_value'].replace(0, np.nan)
).round(4)

print("✅ Feature engineering complete")
print(f"\n   New columns added:")
new_cols = ['delivery_days', 'delivery_delay_days', 'is_late',
            'order_total_value', 'purchase_year', 'purchase_month',
            'purchase_month_name', 'purchase_day_of_week', 'freight_ratio']
for col in new_cols:
    print(f"   • {col}")

✅ Feature engineering complete

   New columns added:
   • delivery_days
   • delivery_delay_days
   • is_late
   • order_total_value
   • purchase_year
   • purchase_month
   • purchase_month_name
   • purchase_day_of_week
   • freight_ratio


In [12]:
# ── Final clean: keep only delivered orders for analysis ────
# Cancelled/unavailable orders have no delivery or price data
# We keep them separately for status analysis

master_all = master.copy()  # keep full version

master_delivered = master[
    master['order_status'] == 'delivered'
].copy()

print("✅ Filtered to delivered orders")
print(f"\n   Order status breakdown:")
print(master['order_status'].value_counts())
print(f"\n   Master (all orders)      : {master_all.shape}")
print(f"   Master (delivered only)  : {master_delivered.shape}")
print(f"\n   Nulls in delivered master:")
print(master_delivered.isnull().sum()[master_delivered.isnull().sum() > 0])

✅ Filtered to delivered orders

   Order status breakdown:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

   Master (all orders)      : (99441, 39)
   Master (delivered only)  : (96478, 39)

   Nulls in delivered master:
order_approved_at                 14
order_delivered_carrier_date       2
order_delivered_customer_date      8
total_payment_value                1
payment_installments               1
payment_type                       1
review_score                     646
review_comment_message           646
review_creation_date             646
delivery_days                      8
delivery_delay_days                8
dtype: int64


In [13]:
# ── Handle remaining nulls in delivered master ──────────────

# Dates (8 rows missing delivery date — drop them, unusable for analysis)
master_delivered = master_delivered.dropna(subset=['order_delivered_customer_date'])

# Payment (1 row missing — drop it)
master_delivered = master_delivered.dropna(subset=['total_payment_value'])

# Review nulls (646 orders have no review — fill with neutral values)
master_delivered['review_score']           = master_delivered['review_score'].fillna(0)
master_delivered['review_comment_message'] = master_delivered['review_comment_message'].fillna('')
master_delivered['review_creation_date']   = master_delivered['review_creation_date'].fillna(pd.NaT)

# order_approved_at (14 nulls — fill with purchase timestamp as approximation)
master_delivered['order_approved_at'] = master_delivered['order_approved_at'].fillna(
    master_delivered['order_purchase_timestamp']
)

# Recalculate delivery_days and delay after dropping bad rows
master_delivered['delivery_days'] = (
    master_delivered['order_delivered_customer_date'] -
    master_delivered['order_purchase_timestamp']
).dt.days

master_delivered['delivery_delay_days'] = (
    master_delivered['order_delivered_customer_date'] -
    master_delivered['order_estimated_delivery_date']
).dt.days

master_delivered['is_late'] = (master_delivered['delivery_delay_days'] > 0).astype(int)

print("✅ Remaining nulls handled")
print(f"   Final shape: {master_delivered.shape}")
print(f"\n   Nulls remaining:")
remaining = master_delivered.isnull().sum()
remaining = remaining[remaining > 0]
print(remaining if len(remaining) > 0 else "   None ✅")

✅ Remaining nulls handled
   Final shape: (96469, 39)

   Nulls remaining:
order_delivered_carrier_date      1
review_creation_date            646
dtype: int64


In [14]:
# ── Final sanity check before saving ───────────────────────
print("=" * 50)
print("  MASTER TABLE — FINAL SUMMARY")
print("=" * 50)
print(f"  Rows        : {master_delivered.shape[0]:,}")
print(f"  Columns     : {master_delivered.shape[1]}")
print(f"\n  Date range  : {master_delivered['order_purchase_timestamp'].min().date()} "
      f"→ {master_delivered['order_purchase_timestamp'].max().date()}")
print(f"\n  Revenue     : R$ {master_delivered['order_total_value'].sum():,.2f}")
print(f"  Avg order   : R$ {master_delivered['order_total_value'].mean():,.2f}")
print(f"\n  Avg delivery: {master_delivered['delivery_days'].mean():.1f} days")
print(f"  Late orders : {master_delivered['is_late'].sum():,} "
      f"({master_delivered['is_late'].mean()*100:.1f}%)")
print(f"\n  Avg review  : {master_delivered[master_delivered['review_score']>0]['review_score'].mean():.2f} / 5")
print(f"\n  Unique customers : {master_delivered['customer_unique_id'].nunique():,}")
print(f"  Unique sellers   : {master_delivered['seller_id'].nunique():,}")
print(f"  Unique categories: {master_delivered['product_category_name_english'].nunique():,}")
print("=" * 50)

  MASTER TABLE — FINAL SUMMARY
  Rows        : 96,469
  Columns     : 39

  Date range  : 2016-10-03 → 2018-08-29

  Revenue     : R$ 15,418,251.37
  Avg order   : R$ 159.83

  Avg delivery: 12.1 days
  Late orders : 6,533 (6.8%)

  Avg review  : 4.16 / 5

  Unique customers : 93,349
  Unique sellers   : 2,960
  Unique categories: 72


## Step 6: Save Clean Master Table
We save two versions:
- `master_all.csv` — all orders including cancelled/shipped (for status analysis)
- `master_delivered.csv` — delivered orders only (for revenue, delivery, and review analysis)

In [15]:
# ── Save clean files ────────────────────────────────────────
output_path = r"C:\Users\yasse\OneDrive\Desktop\data analysis projects\Brazilian-E-Commerce-SQL-POWERBI\Brazilian-Olist-SQL-POWERBI-finished-project\outputs"

os.makedirs(output_path, exist_ok=True)

master_delivered.to_csv(os.path.join(output_path, 'master_delivered.csv'), index=False)
master_all.to_csv(os.path.join(output_path, 'master_all.csv'), index=False)

print("✅ Files saved successfully!")
print(f"\n   📁 master_delivered.csv → {master_delivered.shape[0]:,} rows × {master_delivered.shape[1]} cols")
print(f"   📁 master_all.csv       → {master_all.shape[0]:,} rows × {master_all.shape[1]} cols")
print(f"\n   📂 Saved to: {output_path}")

✅ Files saved successfully!

   📁 master_delivered.csv → 96,469 rows × 39 cols
   📁 master_all.csv       → 99,441 rows × 39 cols

   📂 Saved to: C:\Users\yasse\OneDrive\Desktop\data analysis projects\Brazilian-E-Commerce-SQL-POWERBI\Brazilian-Olist-SQL-POWERBI-finished-project\outputs


## Summary
Phase 1 complete. The master dataset contains **96,469 delivered orders** across **39 columns**,
spanning October 2016 to August 2018.

Key outputs saved to `/outputs/`:
- `master_delivered.csv` — 96,469 rows, delivered orders only
- `master_all.csv` — 99,441 rows, all order statuses

**Next:** Phase 2 — Exploratory Data Analysis (EDA)